# Multi-Generation XRF Screening (M3–M5)

**Speaker notes:**
- Now we analyze multiple generations (M3, M4, M5)
- Emphasize comparison and selection consistency
- Goal: identify best candidates across generations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

In [ ]:
FILE_ID = 'YOUR_FILE_ID'
url = f'https://drive.google.com/uc?id={FILE_ID}'

sheets = ['M3-0 Gy','M3-300 Gy','M4-0 Gy','M4-300 Gy','M5-0 Gy','M5-300 Gy']

dfs = {s: pd.read_excel(url, sheet_name=s) for s in sheets}

for name, d in dfs.items():
    d['Dose'] = 0 if '0 Gy' in name else 300
    d['Generation'] = name.split('-')[0]

df = pd.concat(dfs.values(), ignore_index=True)

df.head()

# Speaker notes:
# - Highlight structure: generation + dose

In [ ]:
WEIGHT_MIN = 2.0
df_clean = df[df['Weight (g)'] >= WEIGHT_MIN]

print('Filtered samples:', len(df_clean))

## Step 1: Examine baseline per generation

**Speaker notes:**
- Each generation may have different baseline
- Must compute threshold separately

In [ ]:
summary = df_clean.groupby(['Generation','Dose'])['Fe ppm calc'].agg(['mean','std','count'])
summary

## Step 2: Compute thresholds per generation

**Speaker notes:**
- Avoid pooling all controls
- Use generation-specific thresholds

In [ ]:
N_SD = 2

thresholds = {}

for gen in ['M3','M4','M5']:
    control = df_clean[(df_clean['Generation']==gen) & (df_clean['Dose']==0)]
    mean = control['Fe ppm calc'].mean()
    std = control['Fe ppm calc'].std()
    thresholds[gen] = mean + N_SD * std

thresholds

# Speaker notes:
# - Show differences between generations

## Step 3: Select mutants per generation

In [ ]:
mutants_list = []

for gen in ['M3','M4','M5']:
    thr = thresholds[gen]
    subset = df_clean[(df_clean['Generation']==gen) & (df_clean['Dose']==300) & (df_clean['Fe ppm calc'] > thr)]
    subset = subset.copy()
    subset['Threshold'] = thr
    mutants_list.append(subset)

mutants = pd.concat(mutants_list)

print('Total mutants:', len(mutants))
mutants.head()

## Step 4: Compare generations

**Speaker notes:**
- Which generation produces more candidates?
- Is effect consistent?

In [ ]:
mutants.groupby('Generation').size()

In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(data=df_clean, x='Generation', y='Fe ppm calc', hue='Dose')
plt.title('Fe distribution across generations')
plt.show()

## Step 5: Multi-element selection (Fe + Zn)

**Speaker notes:**
- Real breeding often uses multiple traits
- Stricter selection

In [ ]:
multi_mutants = []

for gen in ['M3','M4','M5']:
    thr = thresholds[gen]
    subset = df_clean[(df_clean['Generation']==gen) & (df_clean['Dose']==300) & 
                      (df_clean['Fe ppm calc'] > thr) &
                      (df_clean['Zn ppm calc'] > df_clean['Zn ppm calc'].mean())]
    multi_mutants.append(subset)

multi_mutants = pd.concat(multi_mutants)

print('Multi-element mutants:', len(multi_mutants))
multi_mutants.head()

## Interpretation

- Selection differs by generation
- Multi-trait selection reduces candidates

**Speaker notes:**
- Connect to breeding cost and decision-making

In [ ]:
output_file = 'multi_generation_mutants.xlsx'
mutants.to_excel(output_file, index=False)

from google.colab import files
files.download(output_file)